# Create Parquet Files #
This program converts the `.csv` and `.xml` files from Statistics Canada into `.parquet` files. This does not do any pre-processing.

In [1]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import subprocess
import tempfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import shutil

In [2]:
# Define schemas according to native Statistics Canada layouts
CENSUS_CONFIGS = {
    2021: {
        'usecols': [
            'CENSUS_YEAR', 'DGUID', 'GEO_LEVEL', 'GEO_NAME',
            'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME',
            'C1_COUNT_TOTAL', 'C2_COUNT_MEN+', 'C3_COUNT_WOMEN+',
            'C10_RATE_TOTAL', 'C11_RATE_MEN+', 'C12_RATE_WOMEN+'
        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'DGUID': str,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'CHARACTERISTIC_NAME': str,
            'C1_COUNT_TOTAL': np.double,
            'C2_COUNT_MEN+': np.double,
            'C3_COUNT_WOMEN+': np.double,
            'C10_RATE_TOTAL': np.double,
            'C11_RATE_MEN+': np.double,
            'C12_RATE_WOMEN+': np.double,
        },
        'zips': [
            './census_data/98-401-X2021006_eng_CSV.zip',
            './census_data/98-401-X2021010_eng_CSV.zip',
            './census_data/98-401-X2021029_eng_CSV.zip',
            './census_data/98-401-X2021012_eng_CSV.zip'
        ]
    },
    2016: {
        'names': [
            'CENSUS_YEAR', 
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'GNR',
            'GNR_LF',
            'DATA_QUALITY_FLAG',
            'ALT_GEO_CODE',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'NOTES',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'
        ],
        'cols': [
            'CENSUS_YEAR',
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'            

        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'GEO_CODE': np.int64,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'GNR': np.double,
            'GNR_LF': np.double,
            'DATA_QUALITY_FLAG': str,
            'ALT_GEO_CODE': str,
            'CHARACTERISTIC_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'COUNT_TOTAL': np.double,
            'COUNT_MALE': np.double,
            'COUNT_FEMALE': np.double
        },
        'zips': [
            './census_data/98-401-X2016044_eng_CSV.zip',
            './census_data/98-401-X2016045_eng_CSV.zip',
            './census_data/98-401-X2016050_eng_CSV.zip'
        ]
    },
    2011: { # this is for the 2011 NHS data
        'zips': [
            './census_data/99-004-XWE2011001-101_CSV.zip',
            './census_data/99-004-XWE2011001-301_CSV.zip',
            './census_data/99-004-XWE2011001-401_CSV.zip',
            './census_data/99-004-XWE2011001-501_CSV.zip',
            './census_data/99-004-XWE2011001-701_CSV.zip'
        ]
    },
    2010: { # use 2010 for 2011 census data (not NHS)
        'zips': [
            './census_data/98-316-XWE2011001-101_CSV.zip',
            './census_data/98-316-XWE2011001-301_CSV.zip',
            './census_data/98-316-XWE2011001-401_CSV.zip',
            './census_data/98-316-XWE2011001-501_CSV.zip',
            './census_data/98-316-XWE2011001-701_CSV.zip',
            './census_data/98-316-XWE2011001-1501_CSV.zip',            
        ]
    },
    2006: {
        'zips': [
            './census_data/94-581-XCB2006002.ZIP',
            './census_data/94-581-XCB2006005.ZIP',
            './census_data/94-581-XCB2006007.ZIP',
        ]
    },
    2001: {
        'zips': [
            './census_data/95F0495XCB2001002.ZIP',
            './census_data/95F0495XCB2001005.ZIP',
            './census_data/95F0495XCB2001007.ZIP'
        ]
    },
    1996: {
        'zips': [
            './census_data/95F0180XDB96001.ZIP',
            './census_data/95F0181XDB96001.ZIP',
            './census_data/95F0183XDB.ZIP',
            './census_data/95F0185XDB96001.ZIP'
        ]
    },
    1991: {
        'zips': [
            './census_data/1991_Census_Data.zip'
        ]
    },
    1986: {
        'zips': [
            './census_data/1986_Census_Data.zip'
        ]
    },
    1981: {
        'zips': [
            './census_data/1981_Census_Data.zip'
        ]
    }
}

output_dir = "temp_parquet_data"
os.makedirs(output_dir, exist_ok=True)


# Preprocess 2021 Data #

In [ ]:
def process_year_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='latin1',
                        low_memory=False,
                        usecols=year_config['usecols'],
                        dtype=year_config['dtypes'],
                        chunksize=100_000
                    ):

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [5]:
out_file = os.path.join(
    output_dir,
    "2021_raw_census_data.parquet"
)

process_year_data(CENSUS_CONFIGS[2021], out_file)

Processing 98-401-X2021006_English_CSV_data_Atlantic.csv
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,00

# Preprocess 2016 Data #

In [4]:
# Regex patterns for 2016 dynamic characteristic columns
CHAR_NAME_RE = re.compile(r'^DIM:\s*Profile of.*$', re.I)
CHAR_ID_RE = re.compile(r'^Member ID:\s*Profile of.*$', re.I)

def process_2016_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue        

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                if file == 'Geo_starting_row_CSV.csv':
                    continue                

                print(f"Processing {file}")
                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='utf-8-sig',
                        low_memory=False,
                        header=0,
                        names=year_config['names'],
                        dtype=year_config['dtypes'],
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                        chunksize=100_000
                    ):
                        chunk = chunk[year_config['cols']]

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)
                        print(f"  processed {len(chunk):,} rows")

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")


In [5]:
out_file = os.path.join(
    output_dir,
    "2016_raw_census_data.parquet"
)

process_2016_data(CENSUS_CONFIGS[2016], out_file)

Processing 98-401-X2016044_English_CSV_data.csv
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  processed 100,000 rows
  

# Preprocess 2011 Data #

In [21]:
def process_2011_data(year_config, output_path, characteristic_ids, next_characteristic_id):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with tempfile.TemporaryDirectory() as temp_dir:
            subprocess.run(
                ["7z", "x", "-y", f"-o{temp_dir}", zip_path],
                check=True,
                stdout=subprocess.DEVNULL
            )

            for root, _, files in os.walk(temp_dir):
                for file in files:

                    if not file.endswith('.csv'):
                        continue

                    if file.endswith('DQ.csv') or file.endswith('Metadata.csv'):
                        continue

                    file_path = os.path.join(root, file)

                    print(f"Processing {file}")

                    with open(file_path, "rb") as f:
                        # Find where the metadata starts
                        metadata_start_index = None

                        for i, line in enumerate(f):
                            if line.decode("latin1").startswith("Notes:"):
                                metadata_start_index = i
                                break

                        f.seek(0)

                        # sometimes, there is an extra row at the top
                        df = pd.read_csv(f, nrows=1, header=None, encoding='latin1')
                        top_left_value = df.iat[0, 0]

                        if top_left_value != "Geo_Code":
                            # there is a useless row at the top
                            header = 1
                        else:
                            header = 0

                        f.seek(0)
                        """
                        The 2011 census data has different number of columns depending on the geographic level.
                        After analyzing the .csv file structure, I noticed that the name of the geography usually
                        appears one column to the left of the 'Topic' column. 
                        """

                        all_columns = pd.read_csv(
                            f,
                            header=header,
                            nrows=0,
                            encoding='latin1'
                        ).columns.tolist()


                        topic_col_idx = all_columns.index('Topic')

                        geo_index = topic_col_idx - 1
                        has_type_col_flag = False
                        has_gnr_col_flag = False

                        if all_columns[geo_index] == "GNR":
                            has_gnr_col_flag = True
                            # remove the GNR column
                            geo_index -= 1

                        if all_columns[geo_index].lower().endswith("type"):
                            has_type_col_flag = True
                            # type column. No need to keep
                            geo_index -= 1

                        if geo_index > 1:
                            columns_to_skip = [1, geo_index - 1]
                        else:
                            columns_to_skip = []

                        columns_to_keep = [
                            i for i in range(len(all_columns))
                            if i not in columns_to_skip
                        ]

                        # do not need to keep the type column
                        if has_type_col_flag and has_gnr_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)
                            columns_to_keep.remove(topic_col_idx - 2)

                        elif has_type_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)

                        elif has_gnr_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)


                        # do not need topic, note, or flags
                        columns_to_keep.remove(topic_col_idx)
                        # note is two to the right of topic
                        columns_to_keep.remove(topic_col_idx + 2)
                        # flag total is two to the right of topic
                        columns_to_keep.remove(topic_col_idx + 4)
                        # flag male is two to the right of flag total
                        columns_to_keep.remove(topic_col_idx + 6)
                        # flag female is two to the right of male total
                        columns_to_keep.remove(topic_col_idx + 8)
                        # if you were offended that male flag came before female flag, 
                        # you need to calm down.

                        f.seek(0)

                        read_args = {
                            "encoding": "latin1",
                            "low_memory": False,
                            "header": header,
                            "usecols": columns_to_keep,
                            "na_values": ['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                            "chunksize": 10_000
                        }

                        if metadata_start_index is not None:
                            # Header is line 1, data starts at line 2.
                            read_args["nrows"] = metadata_start_index - (1 + header)        

                        for chunk in pd.read_csv(
                            f,
                            **read_args
                        ):

                            # ensure consistent column names
                            chunk.columns = [
                                "GEO_CODE",
                                "GEO_NAME",
                                "CHARACTERISTIC_NAME",
                                "COUNT_TOTAL",
                                "COUNT_MALE",
                                "COUNT_FEMALE"
                            ]

                            chunk = chunk.astype({
                                "GEO_CODE": np.int64,
                                "GEO_NAME": str,
                                "CHARACTERISTIC_NAME": str,
                                "COUNT_TOTAL": np.double,
                                "COUNT_MALE": np.double,
                                "COUNT_FEMALE": np.double
                            })

                            for characteristic in chunk["CHARACTERISTIC_NAME"]:
                                if characteristic not in characteristic_ids:
                                    characteristic_ids[characteristic] = next_characteristic_id
                                    next_characteristic_id += 1

                            chunk["CHARACTERISTIC_ID"] = (
                                chunk["CHARACTERISTIC_NAME"].map(characteristic_ids)
                            )                        

                            schema = pa.schema([
                                ("GEO_CODE", pa.int64()),
                                ("GEO_NAME", pa.string()),
                                ("CHARACTERISTIC_ID", pa.int64()),
                                ("CHARACTERISTIC_NAME", pa.string()),
                                ("COUNT_TOTAL", pa.float64()),
                                ("COUNT_MALE", pa.float64()),
                                ("COUNT_FEMALE", pa.float64()),
                            ])

                            table = pa.Table.from_pandas(
                                chunk,
                                schema=schema,
                                preserve_index=False
                            )

                            if writer is None:
                                writer = pq.ParquetWriter(
                                    output_path,
                                    schema,
                                    compression="snappy"
                                )


                            writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

    return characteristic_ids, next_characteristic_id

In [15]:
def process_2011_da_data(output_path):
    writer = None

    with zipfile.ZipFile('census_data/2011_NHS_DA.zip') as z:
        for file in z.namelist():
            print(f"Processing {file}")

            # Reopen the file for each operation because read_csv consumes it
            with z.open(file) as f:
                all_columns = pd.read_csv(
                    f,
                    nrows=0,
                    encoding='latin1',
                    na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                ).columns.tolist()

            # First column contains the characteristic names
            characteristic_col = all_columns[0]
            geo_columns = all_columns[1:]

            col_chunk_size = 1000

            for i in range(0, len(geo_columns), col_chunk_size):
                current_geo_cols = geo_columns[i:i + col_chunk_size]

                # Reopen the file for every chunk
                with z.open(file) as f:
                    df = pd.read_csv(
                        f,
                        low_memory=False,
                        usecols=[characteristic_col] + current_geo_cols,
                        encoding='latin1',
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL']
                    )

                    # Convert:
                    #
                    # Characteristic | DA001 | DA002
                    # Population     | 100   | 200
                    # Male           | 50    | 100
                    #
                    # into:
                    #
                    # GEO_CODE | CHARACTERISTIC_NAME | VALUE
                    #
                    df = df.melt(
                        id_vars=[characteristic_col],
                        var_name="GEO_CODE",
                        value_name="COUNT_TOTAL"
                    )

                    df['GEO_CODE'] = df['GEO_CODE'].str[0:8]

                    df = df.rename(
                        columns={
                            characteristic_col: "CHARACTERISTIC_NAME"
                        }
                    )

                    df["COUNT_TOTAL"] = pd.to_numeric(
                        df["COUNT_TOTAL"],
                        errors="coerce"
                    ).astype(np.double)

                    schema = pa.schema([
                        ("GEO_CODE", pa.int64()),
                        ("CHARACTERISTIC_NAME", pa.string()),
                        ("COUNT_TOTAL", pa.float64()),
                    ])                    

                    table = pa.Table.from_pandas(
                        df,
                        schema=schema,
                        preserve_index=False
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            output_path,
                            schema,
                            compression='snappy'
                        )

                    writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [22]:
out_file = os.path.join(
    output_dir,
    "2011_raw_census_data.parquet"
)

out_file2 = os.path.join(
    output_dir,
    "2011_raw_nhs_data.parquet"
)

out_file3 = os.path.join(
    output_dir,
    "2011_raw_nhs_da.parquet"
)

characteristic_ids, next_characteristic_id = process_2011_data(CENSUS_CONFIGS[2010], out_file, {}, 0)
process_2011_data(CENSUS_CONFIGS[2011], out_file2, characteristic_ids, next_characteristic_id)
process_2011_da_data(out_file3)

Processing 98-316-XWE2011001-1501-QUE.csv
Processing 98-316-XWE2011001-1501-SASK.csv
Processing 98-316-XWE2011001-1501-NL.csv
Processing 98-316-XWE2011001-1501-PEI.csv
Processing 98-316-XWE2011001-1501-NVT.csv
Processing 98-316-XWE2011001-1501-YT.csv
Processing 98-316-XWE2011001-1501-NB.csv
Processing 98-316-XWE2011001-1501-NS.csv
Processing 98-316-XWE2011001-1501-MAN.csv
Processing 98-316-XWE2011001-1501-ALTA.csv
Processing 98-316-XWE2011001-1501-NWT.csv
Processing 98-316-XWE2011001-1501-ONT.csv
Processing 98-316-XWE2011001-1501-BC.csv
Finished writing: temp_parquet_data/2011_raw_census_data.parquet
Processing 99-004-XWE2011001-101.csv
Processing 99-004-XWE2011001-301-NWT.csv
Processing 99-004-XWE2011001-301-ONT.csv
Processing 99-004-XWE2011001-301-PEI.csv
Processing 99-004-XWE2011001-301-NL.csv
Processing 99-004-XWE2011001-301-BC.csv
Processing 99-004-XWE2011001-301-NS.csv
Processing 99-004-XWE2011001-301-YT.csv
Processing 99-004-XWE2011001-301-QUE.csv
Processing 99-004-XWE2011001-30

# Preprocess 2006 Data #

In [13]:
def process_2006_data(spark, year_config, output_path):
    with tempfile.TemporaryDirectory() as temp_dir:
        parquet_dir = os.path.join(temp_dir, "parquet")

        # Process each ZIP one XML file at a time.
        for zip_path in year_config["zips"]:
            print(f"Processing {zip_path}")

            # Get the XML filenames from the ZIP.
            result = subprocess.run(
                ["unzip", "-Z1", zip_path],
                capture_output=True,
                text=True,
                check=True,
            )

            xml_files = [
                name
                for name in result.stdout.splitlines()
                if name.lower().endswith(".xml")
                and not os.path.basename(name).startswith("Structure")
            ]

            print(f"Found {len(xml_files)} XML files")

            for i, xml_name in enumerate(xml_files, start=1):
                print(f"  [{i}/{len(xml_files)}] {xml_name}")

                xml_path = os.path.join(temp_dir, "input.xml")

                # Extract ONLY this XML file.
                with open(xml_path, "wb") as f:
                    subprocess.run(
                        ["unzip", "-p", zip_path, xml_name],
                        stdout=f,
                        check=True,
                    )

                # Let Spark process this XML.
                df = (
                    spark.read.format("xml")
                    .option("rowTag", "generic:Series")
                    .load(f"file://{xml_path}")
                )

                # Append this XML's rows to the intermediate Parquet dataset.
                df.write \
                    .format("parquet") \
                    .mode("append") \
                    .option("compression", "snappy") \
                    .save(parquet_dir)

                # Remove the huge XML before moving to the next one.
                os.remove(xml_path)

        # Now combine the intermediate Parquet files into one file.
        print("Combining Parquet files...")

        df = spark.read.parquet(parquet_dir)

        temp_output = output_path + "_tmp"

        df.coalesce(1) \
            .write \
            .format("parquet") \
            .mode("overwrite") \
            .option("compression", "snappy") \
            .save(temp_output)

        # Find Spark's single output file.
        part_file = next(
            os.path.join(temp_output, name)
            for name in os.listdir(temp_output)
            if name.startswith("part-") and name.endswith(".parquet")
        )

        # Make sure the destination directory exists.
        os.makedirs(
            os.path.dirname(output_path) or ".",
            exist_ok=True,
        )

        # Move the actual Parquet file to the requested filename.
        if os.path.exists(output_path):
            os.remove(output_path)

        shutil.move(part_file, output_path)

        # Remove Spark's temporary output directory.
        shutil.rmtree(temp_output)

        print(f"Done: {output_path}")

In [12]:
# debuging code (remove later)
with zipfile.ZipFile("./census_data/94-581-XCB2006007.ZIP") as z:
    for name in z.namelist():
        if "Generic_94-581-XCB2006002.xml" in name:
            print(repr(name))

In [14]:
out_file = os.path.join(
    output_dir,
    "2006_raw_census_data.parquet"
)

spark = (
    SparkSession.builder
    .appName("2006 Census")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# this will take a very long time

process_2006_data(spark, CENSUS_CONFIGS[2006], out_file)

Processing ./census_data/94-581-XCB2006002.ZIP
Found 1 XML files
  [1/1] Generic_94-581-XCB2006002.xml


Processing ./census_data/94-581-XCB2006005.ZIP
Found 1 XML files
  [1/1] Generic_94-581-XCB2006005.xml


Processing ./census_data/94-581-XCB2006007.ZIP
Found 1 XML files
  [1/1] Generic_94-581-XCB2006007.xml


Combining Parquet files...


Done: temp_parquet_data/2006_raw_census_data.parquet
